## Part 1: The Tokenizers

### Pre-experiment Predictions

**Which language do you expect to require the most tokens?**
 - Turkish to require the most tokens.
   - Turkish and English both use Latin-based alphabets, but Turkish has _"more productive word morphology"_.
   - Individual Chinese characters are often words on their own.

**How do you expect the larger BPE vocabulary to change the representation of each language?**
 - English — full words or even common short phrases.
 - Turkish — I think it will be the same as English.
 - Chinese — common 2 or more character words and even idioms

**What kinds of units do you expect BPE to learn in English, Turkish, and Chinese?**
 - English — frequently used words (e.g. the, and, a, etc...) and common prefixes and suffixes (e.g. -ing, -er, re-, etc...)
 - Turkish — I think it will be the same as English.
 - Chinese — frequently used characters (e.g. 的，了，个) and common short words.

**Which tokenizer do you expect to work best for language modeling?**
 - I'm guessing the small-sized BPE tokenizer might work best for language modeling.
   - Character-level tokenization will be too slow and inefficient, and making it hard for the model to manage the context.
   - If the BPE vocabulary is too large, there will be a lot more parameters for the model.
   - If the BPE vocabulary is small and compressed at the sweet spot, it is more efficient for the model.


### Pre-experiment Observation

I noticed that the Chinese corpus contains a mixture of traditional and simplified Chinese characters (e.g. 大學 and 大学). Both have the same meaning but would impact the frequencies since they are considered different tokens. This affects evaluations further down the pipeline.

As such, I have decided to do a normalization by converting all the Chinese characters to simplified Chinese characters for the entire dataset.

To run the normalization, run the following command: `uv run ./script/normalize_chinese.py`.


In [ ]:
#!uv run ./script/normalize_chinese.py

### Imports

In [ ]:
import os
import random

import torch
import torch.nn.functional as F
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer
from torch.optim.adamw import AdamW
from torch.utils.data import DataLoader
from tqdm import tqdm

from src import (EOS_ID, LANGUAGES, SPECIAL_TOKENS, UNK_ID, UNK_TOKEN,
                 CharacterTokenizer, DecoderTransformer, LMDataset)

### Set the Device

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda:1')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print(f"Using device: {DEVICE}")

#### Character-level Tokenizer

In [ ]:
character_tokenizer_file = './artifact/tokenizer/character-tokenizer.json'

if os.path.exists(character_tokenizer_file):
    print(f"Tokenizer already exists at {character_tokenizer_file}.")
    character_tokenizer = CharacterTokenizer.from_file(
        character_tokenizer_file
    )
else:
    character_tokenizer = CharacterTokenizer('./data/tokenizer/balanced.txt')
    character_tokenizer.save('./artifact/tokenizer/character-tokenizer.json')

print(character_tokenizer.vocab[:10])

print(
    character_tokenizer.encode('hello').ids,
    character_tokenizer.encode('hello').tokens,
    character_tokenizer.decode(character_tokenizer.encode('hello').ids)
)
print(character_tokenizer.encode("✨").ids, UNK_ID)

#### BPE Tokenizer

In [ ]:
# %%time
def train_bpe(vocab_size, input_file, output_file):
    if os.path.exists(output_file):
        print(f"Tokenizer already exists at {output_file}. Skip training.")
        return Tokenizer.from_file(output_file)

    tokenizer = Tokenizer(BPE(unk_token=UNK_TOKEN))
    trainer = BpeTrainer(
        special_tokens=SPECIAL_TOKENS,
        vocab_size=vocab_size,
        show_progress=False
    )
    tokenizer.pre_tokenizer = Whitespace()
    tokenizer.train([input_file], trainer)
    tokenizer.save(output_file)
    return tokenizer

In [ ]:
bpe_tokenizer_small = train_bpe(
    10000, './data/tokenizer/balanced.txt', './artifact/tokenizer/bpe-tokenizer-small.json')
bpe_tokenizer_big = train_bpe(
    38500, './data/tokenizer/balanced.txt', './artifact/tokenizer/bpe-tokenizer-big.json')

print(f"Small tokenizer's size: {bpe_tokenizer_small.get_vocab_size()}")
print(f"Big tokenizer's size: {bpe_tokenizer_big.get_vocab_size()}")

# print(bpe_tokenizer_small.get_vocab_size(), bpe_tokenizer_big.get_vocab_size())
# print(type(bpe_tokenizer_small.encode('hello').ids))

**Reason for BPE Vocabulary Sizes**

#TODO

#### Analyze the Tokenizers (#TODO)

In [ ]:
encoded = {}

for language in LANGUAGES:
    with open(f'./data/valid/{language}.txt', 'r', encoding='utf-8') as f:
        encoded[language] = bpe_tokenizer_small.encode_batch(
            [line.strip() for line in f])

print(character_tokenizer.get_vocab_size())
print(bpe_tokenizer_small.get_vocab_size())
print(bpe_tokenizer_big.get_vocab_size())

## Part 2: The Language Models

### Prepare the data loaders

In [ ]:
def create_data_loader(languages, tokenizer, context_length, batch_size, type):
    type_label = 'validation' if type == 'valid' else f'{type}ing'

    data = []

    for language in languages:
        with open(f'./data/{type}/{language}.txt', 'r', encoding='utf-8') as f:
            data.extend([line.strip() for line in f])

    print(
        f"Total {type_label} sentences across {len(languages)} languages: {len(data):,d}"
    )

    random.seed(2326)
    random.shuffle(data)
    print(f"Sample {type_label} data: {data[:10]}")

    encodings = tokenizer.encode_batch(data)
    print(encodings[0], encodings[0].ids, encodings[0].tokens)
    token_ids = []

    for encoding in encodings:
        token_ids.extend(encoding.ids)
        token_ids.append(EOS_ID)

    print(
        f"Total {type_label} tokens across {len(languages)} languages: {len(token_ids):,d}")

    dataset = LMDataset(
        tokens=torch.tensor(token_ids),
        context_length=context_length
    )
    data_loader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=True
    )

    return data_loader

In [ ]:
print("="*50)
print("#"*10, "character tokenizer".upper(), "#"*10)
print("-"*50)

training_data_loader_char = create_data_loader(
    languages=LANGUAGES,
    tokenizer=character_tokenizer,
    context_length=128,
    batch_size=32,
    type='train'
)

print()

validation_data_loader_char = create_data_loader(
    languages=LANGUAGES,
    tokenizer=character_tokenizer,
    context_length=128,
    batch_size=32,
    type='valid'
)

print("-"*50)
print("="*50)

print()

print("="*50)
print("#"*10, "bpe (small) tokenizer".upper(), "#"*10)
print("-"*50)

training_data_loader_bpe_small = create_data_loader(
    languages=LANGUAGES,
    tokenizer=bpe_tokenizer_small,
    context_length=128,
    batch_size=32,
    type='train'
)

print()

validation_data_loader_bpe_small = create_data_loader(
    languages=LANGUAGES,
    tokenizer=bpe_tokenizer_small,
    context_length=128,
    batch_size=32,
    type='valid'
)

print("-"*50)
print("="*50)

print()

print("="*50)
print("#"*10, "bpe (big) tokenizer".upper(), "#"*10)
print("-"*50)

training_data_loader_bpe_big = create_data_loader(
    languages=LANGUAGES,
    tokenizer=bpe_tokenizer_big,
    context_length=128,
    batch_size=32,
    type='train'
)

print()

validation_data_loader_bpe_big = create_data_loader(
    languages=LANGUAGES,
    tokenizer=bpe_tokenizer_big,
    context_length=128,
    batch_size=32,
    type='valid'
)

print("-"*50)
print("="*50)

### Training and validation loop

In [ ]:
def train_and_validate(tokenizer, training_data_loader, validation_data_loader):
    model = DecoderTransformer(vocabulary_size=tokenizer.get_vocab_size())
    model = model.to(DEVICE)

    optimizer = AdamW(model.parameters(), lr=0.0001)
    training_loss = []
    validation_loss = []
    NUM_EPOCH = 1

    for epoch in range(NUM_EPOCH):
        model.train()

        running_training_loss = 0.0

        training_progress_bar = tqdm(training_data_loader,
                                     f"[Training] Epoch {epoch + 1} / {NUM_EPOCH}")

        for batch, (x, y) in enumerate(training_progress_bar, 1):
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(x)
            # print(logits.shape, logits.size(2))

            logits_flatten = logits.reshape(-1, logits.size(2))
            targets_flatten = y.reshape(-1)
            # print(targets_flatten.shape, logits_flatten.shape)

            loss = F.cross_entropy(logits_flatten, targets_flatten)
            # print(loss)
            loss.backward()
            optimizer.step()

            running_training_loss += loss.item()
            training_progress_bar.set_postfix(
                batch_loss=f"{loss.item():.4f}", avg_loss=f"{running_training_loss / batch:.4f}")

        model.eval()
        running_validation_loss = 0.0
        validation_progress_bar = tqdm(validation_data_loader,
                                       f"[Validation] Epoch {epoch + 1} / {NUM_EPOCH}")

        with torch.no_grad():
            for batch, (x, y) in enumerate(validation_progress_bar, 1):
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits = model(x)
                logits_flatten = logits.reshape(-1, logits.size(2))
                targets_flatten = y.reshape(-1)
                loss = F.cross_entropy(logits_flatten, targets_flatten)

                running_validation_loss += loss.item()
                training_progress_bar.set_postfix(
                    batch_loss=f"{loss.item():.4f}", avg_loss=f"{running_validation_loss / batch:.4f}")

        epoch_training_loss = running_training_loss/len(training_data_loader)
        training_loss.append(epoch_training_loss)
        epoch_validation_loss = running_validation_loss / \
            len(validation_data_loader)
        validation_loss.append(epoch_validation_loss)

        print(
            f'Average loss for epoch {epoch + 1}: training = {epoch_training_loss:.4f} | validation = {epoch_validation_loss:.4f}\n'
        )

    return model

In [ ]:
print("="*50)
print("#"*10, "character tokenizer".upper(), "#"*10)
print("-"*50)

model_char = train_and_validate(
    character_tokenizer,
    training_data_loader_char,
    validation_data_loader_char
)

torch.save(model_char.state_dict(), "./artifact/model/model_char.pth")

print("-"*50)
print("="*50)

In [ ]:
print("="*50)
print("#"*10, "bpe (small) tokenizer".upper(), "#"*10)
print("-"*50)

model_bpe_small = train_and_validate(
    bpe_tokenizer_small,
    training_data_loader_bpe_small,
    validation_data_loader_bpe_small
)

torch.save(
    model_bpe_small.state_dict(),
    "./artifact/model/model_bpe_small.pth"
)

print("-"*50)
print("="*50)

In [ ]:
# torch.load("./artifact/model/model_bpe_small.pth")

In [ ]:
print("="*50)
print("#"*10, "bpe (big) tokenizer".upper(), "#"*10)
print("-"*50)

model_bpe_big = train_and_validate(
    bpe_tokenizer_big,
    training_data_loader_bpe_big,
    validation_data_loader_bpe_big
)

torch.save(model_bpe_big.state_dict(), "./artifact/model/model_bpe_big.pth")

print("-"*50)
print("="*50)